# Waste detection: training runner for Kaggle

This notebook only **runs** the repo's DVC pipeline on a Kaggle GPU. All the logic lives in the repo.

Setup (right panel):
1. **Accelerator:** GPU T4 x2. `train.py` uses both GPUs through `tf.distribute.MirroredStrategy`. Skip TPU: detection's variable box counts and the NMS decoder make it fiddly, and a T4 is plenty here.
2. **Internet:** ON (pip, git clone, pretrained weights).
3. **Add Input:** your dataset (the folder containing `standardized_384`), plus **Models → Keras / retinanet_resnet50_fpn_coco v4** (pretrained weights; background runs can't auto-download models).
4. **Add-ons > Secrets** (optional): `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` (DVC S3 remote), `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD` (remote MLflow, e.g. DagsHub).

Use **Save Version > Save & Run All (Commit)** for the real run: it keeps running with the browser closed (max 12 h) and keeps `/kaggle/working` as downloadable output.

In [ ]:
REPO = "https://github.com/uniabhi/waste-detection.git"
!git clone -q $REPO /kaggle/working/proj
%cd /kaggle/working/proj
!pip install -q -r requirements.txt
assert _exit_code == 0, "pip install failed (see above)"  # `!` commands don't raise on their own
!nvidia-smi --query-gpu=name,memory.total --format=csv
import tensorflow as tf; print(tf.__version__, tf.config.list_physical_devices('GPU'))

In [ ]:
# First run ever: True = 20 images/class + 1 epoch (~10 min), just to prove the pipeline works.
# Real training run: set False.
SMOKE_TEST = True
if SMOKE_TEST:
    !sed -i 's/limit_per_class: 0 /limit_per_class: 20/; s/^  epochs: 30$/  epochs: 1/' params.yaml
!grep -E "limit_per_class|^  epochs" params.yaml

In [ ]:
# Kaggle inputs are read-only and their path depends on the dataset slug; copy into the repo layout
!mkdir -p archive && cp -r "$(find /kaggle/input -type d -name standardized_384 | head -1)" archive/
!ls archive/standardized_384

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for k in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"]:
    try:
        os.environ[k] = secrets.get_secret(k)
    except Exception:
        print("not set:", k)
HAS_REMOTE = "AWS_ACCESS_KEY_ID" in os.environ

In [ ]:
!test -d .dvc || dvc init -q --no-scm  # repo without `dvc init` still works
# run-cache = DVC remembers (code + params + data) -> outputs, so a stage that already ran
# in an earlier session (e.g. the ~1 h auto-labeling) is restored instead of recomputed
if HAS_REMOTE:
    !pip install -q dvc-s3 boto3  # together, so pip picks a boto3 matching dvc-s3's botocore pin
    !dvc pull --run-cache
!dvc repro
assert _exit_code == 0, "dvc repro failed (see above)"  # else Kaggle would report the run as complete
if HAS_REMOTE:
    !dvc push --run-cache

In [ ]:
!cat reports/autolabel.json reports/metrics.json
# everything you need locally, downloadable from the notebook's Output tab
!zip -qr /kaggle/working/outputs.zip dvc.lock reports models data mlruns
!rm -rf archive  # the image copy would otherwise be saved as notebook output (400 MB, slow to list/download)